# 第007/136节 · 向量、矩阵与批量计算
CPU，依赖torch，种子42。所有输入写在文件中，无需下载数据。
先手算形状和一个点积，再按顺序执行。Notebook包含全部代码，无需lesson.py。
最后在本机重启并运行全部，核对数值。实际验证范围见validation.json。
本节比较固定权重的前向计算，没有训练或性能测试。


## 1. 向量与矩阵的形状
运行前写出各轴的含义和预期输出形状。


In [ ]:
import torch

torch.manual_seed(42)
torch.set_num_threads(1)
device = torch.device("cpu")
# 两行样本，每行三个特征；参数按特征顺序排列。
X = torch.tensor([[1., 2., 3.], [4., 5., 6.]], dtype=torch.float32, device=device)
w = torch.tensor([2., -1., 0.5], dtype=torch.float32, device=device)
w_column = w.reshape(-1, 1)
print("PyTorch", torch.__version__, "device", device)
print("input shapes", tuple(X.shape), tuple(w.shape), tuple(w_column.shape))
print("transpose shape", tuple(X.T.shape))
assert X.shape == (2, 3) and X.T.shape == (3, 2)


## 2. 点积是按元素乘后求和
运行前写出各轴的含义和预期输出形状。


In [ ]:
first_dot = torch.dot(X[0], w)
first_sum = (X[0] * w).sum()
print("first dot", first_dot.item(), "shape", tuple(first_dot.shape))
print("first element products", (X[0] * w).tolist())
assert torch.allclose(first_dot, first_sum)
assert abs(first_dot.item() - 1.5) < 1e-6


## 3. 把多个样本一起计算
运行前写出各轴的含义和预期输出形状。


In [ ]:
per_row = torch.stack([torch.dot(row, w) for row in X])
batch_scores = X @ w
column_scores = X @ w_column
element_products = X * w
print("elementwise shape", tuple(element_products.shape))
print("batch scores", batch_scores.tolist(), "shape", tuple(batch_scores.shape))
print("column scores", column_scores.tolist(), "shape", tuple(column_scores.shape))
assert torch.allclose(batch_scores, per_row)
assert torch.allclose(batch_scores, element_products.sum(dim=1))
assert torch.allclose(column_scores.squeeze(1), batch_scores)
assert batch_scores.shape == (2,) and column_scores.shape == (2, 1)
single_output = batch_scores + 0.25
print("single output with bias", single_output.tolist())
assert torch.allclose(single_output, torch.tensor([1.75, 6.25], device=device))


## 4. 矩阵乘法与多个输出
运行前写出各轴的含义和预期输出形状。


In [ ]:
# W的每一列对应一个输出，两个输出共享同一批输入。
W = torch.tensor([[2., 1.], [-1., 0.], [0.5, -1.]], dtype=torch.float32, device=device)
bias = torch.tensor([0.25, 1.], dtype=torch.float32, device=device)
multi_output = X @ W + bias
by_columns = torch.stack([X @ W[:, j] for j in range(W.shape[1])], dim=1) + bias
print("matrix shapes", tuple(X.shape), tuple(W.shape), tuple(multi_output.shape))
print("multi output", multi_output.tolist())
assert torch.allclose(multi_output, by_columns)
assert torch.allclose(multi_output, torch.tensor([[1.75, -1.], [6.25, -1.]], device=device))
try:
    X @ torch.ones((2, 1), dtype=torch.float32, device=device)
except RuntimeError:
    print("incompatible matmul rejected: inner dimensions 3 and 2")
else:
    raise AssertionError("Expected mismatched inner dimensions to fail")
print("PASS")


## 独立练习
按homework.txt换成三个样本、两个特征，比较逐样本和批量预测。
检查(3,)与(3,1)的区别，再扩展到两个输出。保存个人练习副本。
完成后查看独立参考答案answers.html。
